In [3]:
!pip install iplantuml

In [1]:
import iplantuml

ModuleNotFoundError: No module named 'iplantuml'

# Модули

Модуль это:
* Файл .py
* Бинарные модули, написанные на .C и других языках (DLL или so)

## С точки зрения объектной модели:
* Объект
* Загружается только один инстанс модуля в память
* Отдельное пространство имен (словарь)

Все глобальные переменные и функции, описанные в модуле, становятся видимыми в объекте-модуле при загрузке модуля (import).

## Цели
* Многократное использование кода
* Декомпозиция (разбиение пространства имен системы)
* Реализация разделяемых служб или данных (компоненты, которые в рамках системы представлены в одной копии, синглетон)

Модули удобны для реализации компонентов, которые разделяются в рамках системы и потому требуют только одной копии. Если необходимо предоставить глобальный объект (синглетон), применяемый в некоторых функциях или файлах, тогда его можно реализовать в модуле, который затем будет импортироваться многими клиентами.

## Поиск модуля
* Домашний каталог программы
* Каталог в PYTHONPATH
* Каталоги стандартной библиотеки
* Содержимое любых файлов .pth (при их наличии)
* Подкаталог sites-packages (где размещаются сторонние расширения)

__Список sys.path:__
```
import sys
sys.path
```

__Скомпилированный (бинарный) код модуля:__
* Директория \_\_pycache__ (3.x)
* Файл .pyc (2.x)

In [2]:
%%plantuml
@startuml
Alice -> Bob: Authentication Request
Bob --> Alice: Authentication Response
Alice -> Bob: Another authentication Request
Alice <-- Bob: Another authentication Response
@enduml

UsageError: Cell magic `%%plantuml` not found.


# Использование модулей

* import module
* import module as m
* from module import f
* from module import *

Операция импортирования модуля происходит однократно. Повторное импортирование только создает новую переменную, ссылающуюся на объект модуля.

### Пример (ex1)

__module.py__
```
class F:
    def __init__(self, value):
        self.value = value
    def __str__(self):
        return str(self.value)

def f(arg : F):
    print(__name__ + "." + f.__name__ + "(" + f"{arg}" + ")")
```

__main.py (вариант 1)__
```
import module
module.f(module.F(10))
```

__main.py (вариант 2)__
```
from module import f, F
f(F(10))
```

__main.py (вариант 3)__
```
from module import *
f(F(10))
```

__main.py (вариант 4)__
```
from .module import f, g
g()
```

## Эквивалентность import и from

Код:
```
from module import name1, name2
```

эквивалентен коду:

```
import module
name1 = module.name1
name2 = module.name2
del module
```

## Изменяемые и неизменяемые переменные модулей

__module.py__
```
a = 10
def p():
    print(a)
```

__main.py (версия 1)__:
```
import module
module.a = 123
```

__main.py (версия 1)__:
```
from main import a, p
a = 123
p()
```

## Модуль как синглетон

__globalmodule.py__
```
a = [1, 2, 3]
```

__module.py__
```
import globalmodule

def p():
    print(globalmodule.a)
```

__main.py__
```
import module
import globalmodule

module.p()
globalmodule.a[1] = 100
module.p()
```

## Перезагрузка модуля

__module.py__
```
a = 10
def p():
    print(a)
```

__main.py__
```
import module
#from imp import reload
from importlib import reload

module.p()
module.a = 20
module.p()
module = reload(module)
module.p()

```

# Пакеты

__Что такое пакет:__
* Каталог с кодом python
* Новое пространство имен
* Атрибуты - модули каталога и подкаталоги

__Требования к каталогам:__
* Каждый каталог или подкаталог должен иметь файл \_\_init__.py
* Каждый такой файл может содержать любой код python

__Роли инициализационных файлов пакетов \_\_init\_\_.py:__
* При импортировании пакета первый раз происходит выполнение кода в \_\_init__.py
* Объявление того, что каталог является пакетом
* Инициализация пространства имен пакета
* Поведение оператора from * (переменная \_\_all\_\_)

## Пример. Простой пакет

__Структура директорий__
```
ex.package/
    main.py
    mypkg/
        __init__.py
        modone.py
        modtwo.py
```

__main.py__
```
import mypkg.modone
print(type(mypkg))
print(type(mypkg.modone))
mypkg.modone.g()
```

__modone.py__

```
import modtwo

class A:
    def __init__(self):
        print(__name__ + "." + A.__name__)

def f():
    print(__name__ + "." + f.__name__)

def g():
    modtwo.g2()
```

__modtwo.py__
```
def g2():
    print(__name__ + "." + g.__name__)
```

In [ ]:
%env PYTHONPATH=ex1

## Модель относительного импортирования

Применяется только к модулям пакета. Импортирование осуществляется относительно текущего каталога. Для импортирования других модулей в модулях пакета надо использовать директивы относительного импорта.

Директивы относительного импорта:
* from . import module
* from .. import module
* from .module import func, class
* from ..pkg import module

## Модель пакетов пространств имен (python 3.3 и выше)

Каталоги, содержащие пакеты пространств имен, не содержат \_\_init\_\_.py, в отличие от обычных каталогов пакетов.

__Структура каталогов:__
* ns/dir1/sub/mod1.py
* ns/dir2/sub/mod2.py

__Установка пути поиска:__
set PYTHONPATH=ns/dir1:ns/dir2

__main.py__
```
import sub
print(sub.\_\_path__)
```

# Расширенные возможности модулей

* Модули являются объектами
* Код выполняется всегда в контексте модуля, встроенный модуль \_\_main__ (интерактивный режим)
* Использование префикса _ (_var) для запрета копирования с помощью from *
* Использование переменной \_\_all__ в модуле (определяет имена для импорта)
* Определение контекста выполнения: if \_\_name__ == '\_\_main__': ...
* Модульное тестирование через if \_\_name__
* import \<module\> as \<name\>
* Поддержка обратной совместимости различных версий Python: from \_\_future__ import \<feature\>

# Package Deployment and Distribution (pip3)

Алгоритм создания пакета:
* Подготовить директорию пакета (см. структуру директорий пакета)
* Заполнить конфигурационный файл .toml
* Использовать build backend для создания пакетных файлов (python -m build)
* Зарегистрироваться на индексном сайте (pypi.org, test.pypi.org)
* Загрузить пакет на индексный серверs (twine)

Структура директорий пакета:

```
packagerootdir/
├── LICENSE
├── pyproject.toml
├── README.md
├── src/
│   └── mypackage/
│       ├── __init__.py
│       └── example.py
└── tests/
```


https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#writing-pyproject-toml

# Итог

Модели импортирования (4 штуки):
* Базовые операции импортирования модулей
    * import module
    * from module import attribute
* Операции импортирования пакетов
    * from package.module import attribute
* Операции импортирования относительно пакетов
    * from . import module
* Пакеты пространств имен
    * import <разделенный каталог>.module

# Рабочий проект

Можно контрибьютить (требуется знание git, github)

https://github.com/vitcpp/sphereplot